# Driver Drowsiness Detection Using Real-Time Heart Rate Variability (HRV)

**Portfolio-ready research implementation**

This notebook provides a cleaned and reproducible implementation of the ANN pipeline used for three-class driver-drowsiness detection from HRV features.

**Publication:** *Driver Drowsiness Detection Using Real-Time Heart Rate Variability Data*  
**Model:** Artificial Neural Network (64 → 32 → 3)  
**Classes:** Alert, Early Drowsiness, Severe Drowsiness  
**Published result:** 91.36% accuracy; 85.33% macro precision; 82.33% macro recall; 83.33% macro F1-score.

> **Important labeling note:** The accompanying implementation notebook created labels from HRV thresholds, while the paper also reports Karolinska Sleepiness Scale (KSS) labeling. This refactor supports both approaches. Ground-truth labels are preferred when available; the heuristic mode is included only to reproduce the implementation workflow and should not be treated as independent clinical ground truth.

## 1. Environment

Install the dependencies from the repository root:

```bash
pip install -r requirements.txt
```

Expected raw data columns:

- `participant_full_id` — participant/session identifier (recommended)
- `pulse_rate_bpm` — pulse rate in beats per minute
- `state` — optional ground-truth class label

The loader accepts either `data/pulse_rate_dataset.csv` or multiple CSV files inside `data/`.

In [ ]:
from pathlib import Path
import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

pd.set_option("display.max_columns", 100)

## 2. Configuration

In [ ]:
DATA_DIR = Path("data")
SINGLE_CSV = DATA_DIR / "pulse_rate_dataset.csv"
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PULSE_COL = "pulse_rate_bpm"
PARTICIPANT_COL = "participant_full_id"
TARGET_COL = "state"

# Reproduction-oriented window sizes used by the original implementation.
TIME_WINDOW = 5
FREQ_WINDOW = 20
NONLINEAR_WINDOW = 10
RESAMPLE_HZ = 16.0

TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN = 0.20
EPOCHS = 5
BATCH_SIZE = 32

# "ground_truth" = require a state column in the dataset.
# "heuristic"    = reproduce the threshold-based labeling logic from the
#                  original implementation notebook.
LABEL_MODE = "heuristic"

print(f"TensorFlow version: {tf.__version__}")

## 3. Load and validate the data

In [ ]:
def load_hrv_data(data_dir: Path, single_csv: Path) -> pd.DataFrame:
    if single_csv.exists():
        frame = pd.read_csv(single_csv)
        if PARTICIPANT_COL not in frame.columns:
            frame[PARTICIPANT_COL] = single_csv.stem
        return frame

    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            "No CSV data found. Add data/pulse_rate_dataset.csv or participant CSV files "
            "inside the data/ directory."
        )

    frames = []
    for csv_file in csv_files:
        part = pd.read_csv(csv_file)
        if PARTICIPANT_COL not in part.columns:
            part[PARTICIPANT_COL] = csv_file.stem
        frames.append(part)

    return pd.concat(frames, ignore_index=True)


df_raw = load_hrv_data(DATA_DIR, SINGLE_CSV)

if PULSE_COL not in df_raw.columns:
    raise ValueError(f"Required column '{PULSE_COL}' was not found.")

df_raw[PULSE_COL] = pd.to_numeric(df_raw[PULSE_COL], errors="coerce")
df_raw.loc[df_raw[PULSE_COL] <= 0, PULSE_COL] = np.nan

# Forward-fill within participant to preserve time order; use the training pipeline
# later for feature-level missing-value imputation.
df_raw[PULSE_COL] = (
    df_raw.groupby(PARTICIPANT_COL, sort=False)[PULSE_COL]
    .transform(lambda s: s.ffill())
)
df_raw[PULSE_COL] = df_raw[PULSE_COL].fillna(df_raw[PULSE_COL].median())

print(f"Rows: {len(df_raw):,}")
print(f"Participants/sessions: {df_raw[PARTICIPANT_COL].nunique():,}")
display(df_raw.head())

## 4. Convert pulse rate to RR intervals and perform basic EDA

In [ ]:
df = df_raw.copy()
df["rr_intervals"] = 60000.0 / df[PULSE_COL]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df[PULSE_COL])
ax.set_title("Pulse Rate Over Time")
ax.set_xlabel("Observation")
ax.set_ylabel("Pulse Rate (bpm)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df["rr_intervals"])
ax.set_title("RR Intervals Over Time")
ax.set_xlabel("Observation")
ax.set_ylabel("RR Interval (ms)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. HRV feature engineering

The published study uses:

- **Time-domain:** mean NNI, SDNN, SDSD, NNI50, pNN50, NNI20, pNN20, RMSSD, median NNI, range NNI, CVSD, CVNNI, mean HR, max HR, std HR
- **Frequency-domain:** LF power, HF power, LF/HF ratio
- **Non-linear:** CSI, CVI, modified CSI, sample entropy

Features are computed within each participant/session so that rolling windows do not cross participant boundaries.

In [ ]:
def _safe_std(x, ddof=1):
    x = np.asarray(x, dtype=float)
    if len(x) <= ddof:
        return np.nan
    return float(np.std(x, ddof=ddof))


def time_domain_features(rr_ms: np.ndarray) -> dict:
    rr = np.asarray(rr_ms, dtype=float)
    rr = rr[np.isfinite(rr)]
    if len(rr) == 0:
        return {}

    diffs = np.diff(rr)
    abs_diffs = np.abs(diffs)
    hr = 60000.0 / rr

    mean_nni = float(np.mean(rr))
    sdnn = _safe_std(rr, ddof=1)
    sdsd = _safe_std(diffs, ddof=1) if len(diffs) else np.nan
    rmssd = float(np.sqrt(np.mean(diffs**2))) if len(diffs) else np.nan

    nni_50 = int(np.sum(abs_diffs > 50)) if len(diffs) else 0
    pnni_50 = float(100 * nni_50 / len(diffs)) if len(diffs) else 0.0
    nni_20 = int(np.sum(abs_diffs > 20)) if len(diffs) else 0
    pnni_20 = float(100 * nni_20 / len(diffs)) if len(diffs) else 0.0

    return {
        "mean_nni": mean_nni,
        "sdnn": sdnn,
        "sdsd": sdsd,
        "nni_50": nni_50,
        "pnni_50": pnni_50,
        "nni_20": nni_20,
        "pnni_20": pnni_20,
        "rmssd": rmssd,
        "median_nni": float(np.median(rr)),
        "range_nni": float(np.ptp(rr)),
        "cvsd": float(rmssd / mean_nni) if mean_nni and np.isfinite(rmssd) else np.nan,
        "cvnni": float(sdnn / mean_nni) if mean_nni and np.isfinite(sdnn) else np.nan,
        "mean_hr": float(np.mean(hr)),
        "max_hr": float(np.max(hr)),
        "std_hr": _safe_std(hr, ddof=1),
    }


def frequency_domain_features(rr_ms: np.ndarray, fs: float = RESAMPLE_HZ) -> dict:
    rr = np.asarray(rr_ms, dtype=float)
    rr = rr[np.isfinite(rr)]

    if len(rr) < 5:
        return {"lf_power": np.nan, "hf_power": np.nan, "lf_hf_ratio": np.nan}

    rr_sec = rr / 1000.0
    time_axis = np.cumsum(rr_sec)
    time_axis = time_axis - time_axis[0]

    if time_axis[-1] <= 0:
        return {"lf_power": np.nan, "hf_power": np.nan, "lf_hf_ratio": np.nan}

    resampled_time = np.arange(0, time_axis[-1], 1.0 / fs)
    if len(resampled_time) < 8:
        return {"lf_power": np.nan, "hf_power": np.nan, "lf_hf_ratio": np.nan}

    resampled_rr = np.interp(resampled_time, time_axis, rr_sec)
    resampled_rr = resampled_rr - np.mean(resampled_rr)

    freqs, psd = welch(
        resampled_rr,
        fs=fs,
        nperseg=min(256, len(resampled_rr)),
    )

    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs < 0.40)

    lf_power = float(np.trapz(psd[lf_mask], freqs[lf_mask])) if lf_mask.any() else np.nan
    hf_power = float(np.trapz(psd[hf_mask], freqs[hf_mask])) if hf_mask.any() else np.nan
    ratio = float(lf_power / hf_power) if np.isfinite(hf_power) and hf_power > 0 else np.nan

    return {
        "lf_power": lf_power,
        "hf_power": hf_power,
        "lf_hf_ratio": ratio,
    }


def sample_entropy(signal: np.ndarray, m: int = 2, r_ratio: float = 0.2) -> float:
    x = np.asarray(signal, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) <= m + 1:
        return np.nan

    sd = np.std(x)
    if sd == 0:
        return np.nan
    r = r_ratio * sd

    def match_count(dim: int) -> int:
        templates = np.array([x[i:i + dim] for i in range(len(x) - dim + 1)])
        count = 0
        for i in range(len(templates) - 1):
            distances = np.max(np.abs(templates[i + 1:] - templates[i]), axis=1)
            count += int(np.sum(distances <= r))
        return count

    b = match_count(m)
    a = match_count(m + 1)

    if a == 0 or b == 0:
        return np.nan
    return float(-np.log(a / b))


def nonlinear_features(rr_ms: np.ndarray) -> dict:
    rr = np.asarray(rr_ms, dtype=float)
    rr = rr[np.isfinite(rr)]

    if len(rr) < 3:
        return {"csi": np.nan, "cvi": np.nan, "modified_csi": np.nan, "sampen": np.nan}

    diffs = np.diff(rr)
    sd1_sq = 0.5 * np.var(diffs)
    sd2_sq = 2.0 * np.var(rr) - 0.5 * np.var(diffs)

    if sd1_sq <= 0 or sd2_sq <= 0:
        csi = cvi = modified_csi = np.nan
    else:
        sd1 = np.sqrt(sd1_sq)
        sd2 = np.sqrt(sd2_sq)
        csi = sd2 / sd1
        cvi = np.log10(16.0 * sd1 * sd2)
        modified_csi = 4.0 * (sd2**2) / sd1

    return {
        "csi": float(csi) if np.isfinite(csi) else np.nan,
        "cvi": float(cvi) if np.isfinite(cvi) else np.nan,
        "modified_csi": float(modified_csi) if np.isfinite(modified_csi) else np.nan,
        "sampen": sample_entropy(rr),
    }

In [ ]:
def rolling_feature_table(participant_df: pd.DataFrame) -> pd.DataFrame:
    rr = participant_df["rr_intervals"].to_numpy(dtype=float)
    records = []

    for i in range(len(participant_df)):
        time_start = max(0, i - TIME_WINDOW + 1)
        freq_start = max(0, i - FREQ_WINDOW + 1)
        nonlinear_start = max(0, i - NONLINEAR_WINDOW + 1)

        record = {}
        record.update(time_domain_features(rr[time_start:i + 1]))
        record.update(frequency_domain_features(rr[freq_start:i + 1]))
        record.update(nonlinear_features(rr[nonlinear_start:i + 1]))
        records.append(record)

    return pd.DataFrame(records, index=participant_df.index)


feature_frames = []
for _, participant_df in df.groupby(PARTICIPANT_COL, sort=False):
    feature_frames.append(rolling_feature_table(participant_df))

features = pd.concat(feature_frames).sort_index()
df_features = pd.concat(
    [df[[PARTICIPANT_COL, PULSE_COL, "rr_intervals"]], features],
    axis=1,
)

print(f"Engineered feature table shape: {df_features.shape}")
display(df_features.head(10))

### Inspect Power Spectral Density (example window)

In [ ]:
example_rr = (
    df.groupby(PARTICIPANT_COL, sort=False)["rr_intervals"]
    .apply(lambda s: s.iloc[: min(len(s), FREQ_WINDOW)].to_numpy())
    .iloc[0]
)

if len(example_rr) >= 5:
    rr_sec = example_rr / 1000.0
    time_axis = np.cumsum(rr_sec)
    time_axis -= time_axis[0]
    resampled_time = np.arange(0, time_axis[-1], 1.0 / RESAMPLE_HZ)
    resampled_rr = np.interp(resampled_time, time_axis, rr_sec)
    resampled_rr -= np.mean(resampled_rr)

    freqs, psd = welch(
        resampled_rr,
        fs=RESAMPLE_HZ,
        nperseg=min(256, len(resampled_rr)),
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.semilogy(freqs, psd, label="PSD")
    ax.axvspan(0.04, 0.15, alpha=0.20, label="LF band (0.04–0.15 Hz)")
    ax.axvspan(0.15, 0.40, alpha=0.20, label="HF band (0.15–0.40 Hz)")
    ax.set_xlim(0, min(2.0, RESAMPLE_HZ / 2))
    ax.set_title("Power Spectral Density of HRV")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power Spectral Density")
    ax.legend()
    plt.show()

## 6. Prepare class labels

For a strong research repository, use independently collected ground-truth labels whenever possible.

- `LABEL_MODE = "ground_truth"` uses an existing `state` column.
- `LABEL_MODE = "heuristic"` reproduces the threshold-based logic found in the original implementation notebook.

The heuristic labels are generated from model input features themselves, so they can create a circular prediction task. They are useful for implementation reproduction, not as a substitute for independently measured KSS labels.

In [ ]:
def heuristic_state(row: pd.Series) -> str:
    rmssd = row["rmssd"]
    lf = row["lf_power"]
    hf = row["hf_power"]
    sampen = row["sampen"]

    # Reproduction of the original implementation logic.
    if (
        np.isfinite(rmssd) and np.isfinite(lf) and np.isfinite(hf) and np.isfinite(sampen)
        and rmssd < 200
        and lf < 0.008
        and hf < 0.008
        and sampen < 1.5
    ):
        return "alert"

    if (
        (np.isfinite(rmssd) and 200 < rmssd < 300)
        or (np.isfinite(lf) and 0.008 <= lf < 0.01)
        or (np.isfinite(hf) and 0.008 <= hf < 0.01)
        or (np.isfinite(sampen) and 1.2 <= sampen < 2.0)
    ):
        return "early drowsiness"

    return "severe drowsiness"


if LABEL_MODE == "ground_truth":
    if TARGET_COL not in df_raw.columns:
        raise ValueError(
            f"LABEL_MODE='ground_truth' requires a '{TARGET_COL}' column. "
            "The paper text supplied with this project does not specify numeric KSS cutoffs, "
            "so this notebook does not invent them."
        )
    df_features[TARGET_COL] = df_raw[TARGET_COL].astype(str).str.strip().str.lower()

elif LABEL_MODE == "heuristic":
    df_features[TARGET_COL] = df_features.apply(heuristic_state, axis=1)

else:
    raise ValueError("LABEL_MODE must be either 'ground_truth' or 'heuristic'.")

class_counts = df_features[TARGET_COL].value_counts()
print(class_counts)

## 7. Build train, validation, and test sets

In [ ]:
FEATURE_COLUMNS = [
    "mean_nni", "sdnn", "sdsd", "nni_50", "pnni_50",
    "nni_20", "pnni_20", "rmssd", "median_nni", "range_nni",
    "cvsd", "cvnni", "mean_hr", "max_hr", "std_hr",
    "lf_power", "hf_power", "lf_hf_ratio",
    "csi", "cvi", "modified_csi", "sampen",
]

X = df_features[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
y_text = df_features[TARGET_COL].astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE_WITHIN_TRAIN,
    random_state=SEED,
    stratify=y_train_full,
)

# Fit preprocessing only on training data to avoid data leakage.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight = dict(zip(classes, weights))

print("Class names:", list(label_encoder.classes_))
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)
print("Class weights:", class_weight)

## 8. Train the ANN model

In [ ]:
def build_ann(input_dim: int, n_classes: int) -> tf.keras.Model:
    model = Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(64, activation="relu", name="dense_64"),
            Dense(32, activation="relu", name="dense_32"),
            Dense(n_classes, activation="softmax", name="drowsiness_output"),
        ],
        name="hrv_drowsiness_ann",
    )
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_ann(
    input_dim=X_train_scaled.shape[1],
    n_classes=len(label_encoder.classes_),
)

model.summary()

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    verbose=1,
)

## 9. Evaluate the model

In [ ]:
test_prob = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(test_prob, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

results = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"],
        "Score": [accuracy, precision, recall, f1],
    }
)
display(results.style.format({"Score": "{:.4f}"}))

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0,
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=label_encoder.classes_,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Confusion Matrix")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Training curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history.history["accuracy"], marker="o", label="Training")
ax.plot(history.history["val_accuracy"], marker="o", label="Validation")
ax.set_title("Training and Validation Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history.history["loss"], marker="o", label="Training")
ax.plot(history.history["val_loss"], marker="o", label="Validation")
ax.set_title("Training and Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 11. Save reusable artifacts

In [ ]:
model.save(ARTIFACT_DIR / "hrv_drowsiness_ann.keras")
joblib.dump(imputer, ARTIFACT_DIR / "imputer.joblib")
joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")
joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")

metadata = {
    "feature_columns": FEATURE_COLUMNS,
    "label_mode": LABEL_MODE,
    "seed": SEED,
    "time_window": TIME_WINDOW,
    "frequency_window": FREQ_WINDOW,
    "nonlinear_window": NONLINEAR_WINDOW,
    "resample_hz": RESAMPLE_HZ,
}
joblib.dump(metadata, ARTIFACT_DIR / "metadata.joblib")

print(f"Saved model and preprocessing objects to: {ARTIFACT_DIR.resolve()}")

## 12. Research reproducibility notes

1. **Published metrics vs. rerun metrics:** The publication reports 91.36% accuracy, 85.33% precision, 82.33% recall, and 83.33% F1-score. A rerun can differ because this refactor prevents preprocessing leakage and uses a separate validation set.
2. **Class imbalance:** The paper reports 96 alert, 544 early-drowsiness, and 167 severe-drowsiness instances. Balanced class weights are used here during training.
3. **Ground truth:** Prefer KSS/experiment labels over feature-derived heuristic labels.
4. **Participant leakage:** For a stricter generalization test, split by participant rather than by row. That evaluates performance on completely unseen drivers.
5. **Data privacy:** Do not publish participant-level raw physiological data unless the consent/ethics/data-sharing conditions allow it.